Almost every real program reads from or writes to files — configuration, data, logs, exports. Python makes file I/O straightforward but has enough subtleties — text vs binary mode, encodings, line endings, resource cleanup — that it is worth understanding properly. This notebook covers the built-in `open()` function, context managers for safe resource handling, the modern `pathlib` module for path manipulation, and working with the two most common structured file formats: CSV and JSON.

## Opening Files — `open()`

`open(path, mode, encoding)` returns a file object. The most important parameter after the path is the **mode**:

| Mode | Meaning |
|------|----------|
| `"r"` | Read text (default) |
| `"w"` | Write text — creates or **truncates** |
| `"a"` | Append text — creates or adds to end |
| `"x"` | Exclusive create — fails if file exists |
| `"r+"` | Read and write |
| `"rb"`, `"wb"` | Binary read / write |

Always close files after use. The `with` statement does this automatically — even when an exception occurs. It is the standard, correct way to work with files.

In [ ]:
import tempfile, os

# Use a temp directory so we don't litter the working directory
TMP = tempfile.mkdtemp()
SAMPLE = os.path.join(TMP, "sample.txt")

# Write a file
with open(SAMPLE, "w", encoding="utf-8") as f:
    f.write("Line one\n")
    f.write("Line two\n")
    f.write("Line three\n")

print(f"Wrote to: {SAMPLE}")

# Read the whole file as a single string
with open(SAMPLE, "r", encoding="utf-8") as f:
    content = f.read()

print(repr(content))

## Reading Files

File objects provide several ways to read:

- **`f.read()`** — reads the entire file into one string.
- **`f.readline()`** — reads one line (including the trailing `\n`).
- **`f.readlines()`** — reads all lines into a list.
- **Iterating the file object** — yields one line per iteration, memory-efficient for large files.

In [ ]:
# Iterate line by line — best for large files (no full read into memory)
with open(SAMPLE, encoding="utf-8") as f:
    for lineno, line in enumerate(f, start=1):
        print(f"{lineno:3}: {line}", end="")   # line already has \n

print()

# readlines() — all lines as a list
with open(SAMPLE, encoding="utf-8") as f:
    lines = f.readlines()

print(lines)   # ['Line one\n', 'Line two\n', 'Line three\n']

# strip() removes the trailing newline from each line
clean = [line.rstrip("\n") for line in lines]
print(clean)

In [ ]:
# f.tell() and f.seek() — position within the file
with open(SAMPLE, encoding="utf-8") as f:
    first_line = f.readline()
    position   = f.tell()       # byte offset after first line
    second_line = f.readline()

    f.seek(0)                   # rewind to start
    again = f.readline()

print(repr(first_line))
print(f"Position after first line: {position}")
print(repr(second_line))
print(repr(again))   # same as first_line

## Writing Files

In [ ]:
LOG = os.path.join(TMP, "app.log")

# writelines() writes an iterable of strings — does NOT add newlines automatically
entries = ["INFO: server started\n", "INFO: listening on :8080\n"]
with open(LOG, "w", encoding="utf-8") as f:
    f.writelines(entries)

# Append mode — adds to the end, does not truncate
with open(LOG, "a", encoding="utf-8") as f:
    f.write("WARNING: high memory usage\n")

with open(LOG, encoding="utf-8") as f:
    print(f.read())

In [ ]:
# print() can write to a file — convenient for formatted output
REPORT = os.path.join(TMP, "report.txt")

data = [("Alice", 92), ("Bob", 75), ("Carol", 88)]

with open(REPORT, "w", encoding="utf-8") as f:
    print("=" * 30, file=f)
    print("Student Report", file=f)
    print("=" * 30, file=f)
    for name, score in data:
        print(f"{name:<10} {score:>5}", file=f)

with open(REPORT, encoding="utf-8") as f:
    print(f.read())

## Text vs Binary Mode

In **text mode** (`"r"`, `"w"`), Python decodes bytes to strings using an encoding (default: platform-dependent — always specify `encoding="utf-8"` explicitly). It also translates line endings: `\r\n` on Windows is normalised to `\n` on reading.

In **binary mode** (`"rb"`, `"wb"`), Python reads and writes raw `bytes` with no encoding or line-ending translation. Use binary mode for images, audio, executables, or any non-text format.

In [ ]:
BIN = os.path.join(TMP, "data.bin")

# Write raw bytes
with open(BIN, "wb") as f:
    f.write(b"\x89PNG\r\n\x1a\n")   # first 8 bytes of every PNG file
    f.write(bytes(range(16)))

# Read raw bytes
with open(BIN, "rb") as f:
    header = f.read(8)
    rest   = f.read()

print(header)
print(list(rest))

# Encoding awareness in text mode
UNICODE = os.path.join(TMP, "unicode.txt")
with open(UNICODE, "w", encoding="utf-8") as f:
    f.write("Héllo, 世界! 🐍\n")

with open(UNICODE, "r", encoding="utf-8") as f:
    print(f.read())

# Reading the same file as bytes shows the raw UTF-8 encoding
with open(UNICODE, "rb") as f:
    raw = f.read()
print(raw)

## `pathlib` — Modern Path Handling

The `pathlib` module (Python 3.4+) represents file system paths as objects rather than plain strings. `Path` objects know how to be joined, split, tested, and iterated — operations that would otherwise require `os.path` calls and string manipulation.

Use `pathlib` for all new code. It is cleaner, more readable, and cross-platform.

In [ ]:
from pathlib import Path

# Create a Path — works on Windows, macOS, and Linux
p = Path(TMP)

print(p)                      # /tmp/...
print(p.name)                 # directory name
print(p.parent)               # parent directory
print(p.is_dir())             # True

# / operator joins paths — works on all platforms
config_path = p / "config" / "settings.json"
print(config_path)
print(config_path.suffix)     # .json
print(config_path.stem)       # settings
print(config_path.name)       # settings.json
print(config_path.parent)     # .../config

In [ ]:
# Common path predicates
sample = Path(SAMPLE)

print(sample.exists())        # True
print(sample.is_file())       # True
print(sample.is_dir())        # False
print(sample.stat().st_size)  # file size in bytes

# Resolve to absolute path, resolving symlinks
print(sample.resolve())

# Change suffix
backup = sample.with_suffix(".bak")
print(backup)

# Change name entirely
renamed = sample.with_name("output.txt")
print(renamed)

In [ ]:
# Read and write directly from Path — no open() needed
notes = Path(TMP) / "notes.txt"

notes.write_text("First note\nSecond note\n", encoding="utf-8")
print(notes.read_text(encoding="utf-8"))

# Binary equivalent
img = Path(TMP) / "icon.bin"
img.write_bytes(bytes(range(256)))
data = img.read_bytes()
print(f"Read {len(data)} bytes")

### Directory Operations

In [ ]:
base = Path(TMP) / "project"

# mkdir — parents=True creates intermediate dirs; exist_ok=True won't raise if it exists
(base / "src").mkdir(parents=True, exist_ok=True)
(base / "tests").mkdir(parents=True, exist_ok=True)
(base / "docs").mkdir(parents=True, exist_ok=True)

# Create some files
(base / "src" / "main.py").write_text("print('hello')\n")
(base / "src" / "utils.py").write_text("def helper(): pass\n")
(base / "tests" / "test_main.py").write_text("import pytest\n")
(base / "README.md").write_text("# My Project\n")

# iterdir() — immediate children
print("Top-level contents:")
for item in sorted(base.iterdir()):
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"  {kind}  {item.name}")

In [ ]:
# glob() — pattern matching within a directory
print("All Python files (top level only):")
for p in sorted(base.glob("*.py")):
    print(f"  {p.name}")

# rglob() — recursive glob (searches all subdirectories)
print("\nAll Python files (recursive):")
for p in sorted(base.rglob("*.py")):
    print(f"  {p.relative_to(base)}")

# All files recursively
print("\nAll files:")
for p in sorted(base.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(base)}")

In [ ]:
# Rename and delete
src_file = base / "README.md"
dst_file = base / "README.txt"

src_file.rename(dst_file)   # moves/renames the file
print(dst_file.exists())    # True
print(src_file.exists())    # False

dst_file.unlink()           # delete a file
print(dst_file.exists())    # False

# rmdir() only works on empty directories
# For a non-empty tree use shutil.rmtree
import shutil
shutil.rmtree(base / "docs")   # removes docs/ and all its contents
print((base / "docs").exists())

## CSV Files

CSV (comma-separated values) is the most common plain-text data interchange format. Python's `csv` module handles the quoting, escaping, and dialect differences correctly — do not parse CSV by hand with `.split(",")`.

In [ ]:
import csv

CSV_FILE = os.path.join(TMP, "employees.csv")

# Writing CSV
employees = [
    {"name": "Alice",  "role": "engineer",  "salary": 95000},
    {"name": "Bob",    "role": "designer",   "salary": 80000},
    {"name": "Carol",  "role": "manager",    "salary": 110000},
    {"name": "Dave",   "role": "engineer",   "salary": 92000},
]

with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
    # DictWriter uses field names from the dict keys
    writer = csv.DictWriter(f, fieldnames=["name", "role", "salary"])
    writer.writeheader()
    writer.writerows(employees)

# Verify the raw content
print(Path(CSV_FILE).read_text())

In [ ]:
# Reading CSV with DictReader — each row is an OrderedDict
with open(CSV_FILE, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Columns: {reader.fieldnames}")

for row in rows:
    print(f"  {row['name']:<8} {row['role']:<10} ${int(row['salary']):,}")

# Values are strings — convert to the right type as needed
salaries = [int(row["salary"]) for row in rows]
print(f"\nAvg salary: ${sum(salaries) / len(salaries):,.0f}")

In [ ]:
# newline="" is important — let the csv module handle line endings
# open() in text mode translates \r\n to \n, which can confuse csv on Windows
# Passing newline="" disables that translation and lets csv.reader handle it

# Custom delimiter (tab-separated)
TSV = os.path.join(TMP, "data.tsv")
with open(TSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerows([("x", "y"), (1, 2), (3, 4)])

with open(TSV, newline="", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="\t"):
        print(row)

## JSON Files

JSON is the standard format for structured data exchange — APIs, configuration files, logs. Python's `json` module converts between Python objects and JSON text.

| Python | JSON |
|--------|------|
| `dict` | object `{}` |
| `list`, `tuple` | array `[]` |
| `str` | string |
| `int`, `float` | number |
| `True`, `False` | `true`, `false` |
| `None` | `null` |

In [ ]:
import json

JSON_FILE = os.path.join(TMP, "config.json")

config = {
    "app": "MyService",
    "version": "2.1.0",
    "debug": False,
    "database": {
        "host": "localhost",
        "port": 5432,
        "name": "production",
    },
    "allowed_hosts": ["example.com", "api.example.com"],
    "max_connections": None,
}

# json.dump — write to file
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)   # indent makes it human-readable

print(Path(JSON_FILE).read_text())

In [ ]:
# json.load — read from file
with open(JSON_FILE, encoding="utf-8") as f:
    loaded = json.load(f)

print(type(loaded))                         # dict
print(loaded["database"]["port"])           # 5432
print(loaded["debug"])                      # False
print(loaded["max_connections"])            # None

# json.dumps / json.loads — work with strings instead of files
text = json.dumps({"key": "value", "n": 42})
print(text)                                 # {"key": "value", "n": 42}

data = json.loads('{"x": 1, "active": true}')
print(data, type(data["active"]))           # bool, not str

In [ ]:
# Handling types json doesn't support natively — custom encoder
import datetime
from decimal import Decimal

class ExtendedEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime.datetime):
            return obj.isoformat()
        if isinstance(obj, Decimal):
            return float(obj)
        return super().default(obj)


event = {
    "name": "Conference",
    "date": datetime.datetime(2026, 6, 15, 9, 0),
    "price": Decimal("199.99"),
}

print(json.dumps(event, cls=ExtendedEncoder, indent=2))

## In-Memory Files — `io.StringIO` and `io.BytesIO`

`io.StringIO` and `io.BytesIO` are file-like objects backed by an in-memory buffer rather than disk. They support the full file API — `read()`, `write()`, `seek()`, `tell()` — and are accepted anywhere a file object is expected.

Common uses: testing code that writes to files without touching the disk, building a string by writing to it incrementally, parsing a string with a library that expects a file object.

In [ ]:
import io

# Write CSV to a string in memory instead of disk
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=["name", "score"])
writer.writeheader()
writer.writerows([{"name": "Alice", "score": 92}, {"name": "Bob", "score": 85}])

csv_string = buf.getvalue()
print(repr(csv_string))

# Read it back with csv.DictReader
reader = csv.DictReader(io.StringIO(csv_string))
for row in reader:
    print(row)

In [ ]:
# BytesIO — useful for testing image/binary processing code
buf = io.BytesIO()
buf.write(b"HEADER")
buf.write(bytes(range(10)))

buf.seek(0)                   # rewind before reading
header = buf.read(6)
payload = buf.read()

print(header)                 # b'HEADER'
print(list(payload))          # [0, 1, 2, ..., 9]
print(f"Total: {len(buf.getvalue())} bytes")

## Practical Patterns

A few everyday patterns that combine what we have covered.

In [ ]:
# Safe atomic write — write to a temp file, then rename
# rename() is atomic on POSIX — readers never see a half-written file
import tempfile

def atomic_write(path: Path, content: str, encoding: str = "utf-8") -> None:
    """Write content to path atomically via a temporary file."""
    tmp = path.with_suffix(".tmp")
    try:
        tmp.write_text(content, encoding=encoding)
        tmp.rename(path)
    except Exception:
        tmp.unlink(missing_ok=True)   # clean up temp on failure
        raise


target = Path(TMP) / "important.json"
atomic_write(target, json.dumps({"status": "ok"}, indent=2))
print(target.read_text())

In [ ]:
# Load JSON with a fallback default if the file is missing
def load_json(path: Path, default=None):
    """Load JSON from path, returning default if the file does not exist."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError:
        return default
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path}: {e}") from e


data = load_json(Path(TMP) / "config.json", default={})
missing = load_json(Path(TMP) / "does_not_exist.json", default={"fallback": True})
print(data["app"])
print(missing)

In [ ]:
# Count lines, words, and characters — like Unix wc
def word_count(path: Path) -> dict:
    text  = path.read_text(encoding="utf-8")
    lines = text.splitlines()
    words = text.split()
    chars = len(text)
    return {"lines": len(lines), "words": len(words), "chars": chars}


for py_file in sorted((Path(TMP) / "project").rglob("*.py")):
    wc = word_count(py_file)
    print(f"  {py_file.name:<20} {wc['lines']:>3}L  {wc['words']:>4}W  {wc['chars']:>5}C")

## Summary

- **`open(path, mode, encoding)`**: always use a `with` statement to ensure the file is closed. Modes: `"r"` (read), `"w"` (write/truncate), `"a"` (append), `"x"` (exclusive create), `"rb"`/`"wb"` (binary). Always specify `encoding="utf-8"` for text mode.
- **Reading**: `f.read()` (whole file), `f.readline()` (one line), `f.readlines()` (list of lines), or iterate the file object (memory-efficient). Use `f.seek(0)` to rewind.
- **Writing**: `f.write(str)` and `f.writelines(iterable)`. `print(..., file=f)` works too. Open with `"a"` to append without truncating.
- **Text mode** decodes bytes to str and normalises line endings. **Binary mode** reads/writes raw `bytes` — use for non-text formats. Always specify `encoding="utf-8"` explicitly.
- **`pathlib.Path`**: cross-platform path objects. Use `/` to join, `.name`/`.stem`/`.suffix`/`.parent` to inspect. `.exists()`, `.is_file()`, `.is_dir()` for tests. `.read_text()`, `.write_text()`, `.read_bytes()`, `.write_bytes()` for direct I/O. `.mkdir(parents=True, exist_ok=True)` to create directories. `.glob()` and `.rglob()` for pattern matching. `.rename()`, `.unlink()`, and `shutil.rmtree()` for modification.
- **CSV**: use `csv.DictWriter`/`csv.DictReader`. Always open with `newline=""` and specify `encoding`. Values are always strings on read — convert types explicitly.
- **JSON**: `json.dump`/`json.load` for files; `json.dumps`/`json.loads` for strings. Use `indent=` for human-readable output. Subclass `json.JSONEncoder` to handle non-serialisable types like `datetime` and `Decimal`.
- **`io.StringIO`/`io.BytesIO`**: in-memory file objects — same API as real files. Use for testing, building strings incrementally, and passing string data to file-expecting APIs.